# Ablation Study: Architecture C without S Model

This notebook investigates the impact of removing the smallest developer model (S tier) from Architecture C.

## Motivation

In Architecture C, tasks are routed to different developer tiers (S/M/L) based on story points:
- **S tier** (1-2 story points): `Qwen2.5-Coder-1.5B-Instruct`
- **M tier** (3-5 story points): `Qwen2.5-Coder-7B-Instruct`  
- **L tier** (8 story points): `Qwen2.5-Coder-32B-Instruct`

However, observations suggest that the S tier model is rarely used effectively, as tasks often escalate to larger models anyway. This ablation study compares:

| Architecture | S Tier Model | M Tier Model | L Tier Model |
|-------------|--------------|--------------|---------------|
| C (baseline) | 1.5B | 7B | 32B |
| C2 (no S) | 7B (same as M) | 7B | 32B |

**Research Question**: Does removing the S tier affect pass rate or efficiency?

---
## Setup

In [1]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

Cloning into '/content/ArchitecturesForCodeDevelopmentWithLLMs'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.4/536.4 k

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai

In [2]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

# Configurazione LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "project_name"
os.environ["LANGCHAIN_API_KEY"] = "langchain_api_key"

# Forza sempre l'uso del token che inserisci
os.environ["HF_TOKEN"] = "hf_token"
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

os.environ["ARCHITECTURE"] = "C2"
print("ARCHITECTURE set to", os.environ["ARCHITECTURE"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


ARCHITECTURE set to C2


In [4]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

Logged in to Hugging Face as: Riaburger


In [5]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("ablation_c2")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "ablation_c2.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)

2026-01-28 09:42:07,687 | INFO | Logger ready. Repo root: /content/ArchitecturesForCodeDevelopmentWithLLMs


---
## Ablation Study: Run Architecture C2 (No S Model)

In [6]:
import time
import json
import random
from src.data.task_loader import HumanEvalTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

# Use C2 architecture (no S tier - maps S to M model)
ARCH = Architecture.C2
SEED_FIXED = 31

def run_humaneval_benchmark(limit: int = 15, shuffle: bool = True):
    """
    Run benchmark on HumanEval tasks with Architecture C2.
    
    Args:
        limit: Number of tasks to run
        shuffle: If True, randomly sample tasks with fixed seed
    """
    loader = HumanEvalTaskLoader()
    all_tasks = loader.load_all()
    
    if shuffle:
        random.seed(SEED_FIXED)
        tasks = random.sample(all_tasks, min(limit, len(all_tasks)))
    else:
        tasks = all_tasks[:limit]

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks from HumanEval (Architecture C2 - No S Model)", total)
    
    for i, task in enumerate(tasks, 1):
        task_id = task.task_id
        logger.info("[%d/%d] Running task %s", i, total, task_id)
        
        start_time = time.time()
        try:
            final_state = run_graph(
                task_id=task_id,
                task_description=task.prompt,
                test_code=task.test,
                entry_point=task.entry_point,
                architecture=ARCH
            )
            
            elapsed = time.time() - start_time
            result = {
                "task_id": task_id,
                "architecture": ARCH.value,
                "test_passed": final_state["test_passed"],
                "escalations": final_state["escalations"],
                "story_points_initial": final_state["story_points_initial"],
                "story_points_current": final_state["story_points_current"],
                "developer_tier": final_state["developer_tier"],
                "elapsed_seconds": round(elapsed, 2),
                "generated_code": final_state["generated_code"],
                "failure_history": final_state["failure_history"],
                "reviewer_feedback": final_state.get("reviewer_feedback", ""),
            }
            results.append(result)
            
            status = "PASS" if final_state["test_passed"] else "FAIL"
            logger.info("[%d/%d] %s %s in %.2fs (tier=%s, escalations=%d)",
                       i, total, task_id, status, elapsed,
                       final_state["developer_tier"], final_state["escalations"])
            
        except Exception as e:
            elapsed = time.time() - start_time
            logger.error("[%d/%d] %s ERROR: %s", i, total, task_id, str(e))
            results.append({
                "task_id": task_id,
                "architecture": ARCH.value,
                "test_passed": False,
                "error": str(e),
                "elapsed_seconds": round(elapsed, 2),
            })
    
    # Save results
    output_file = LOG_DIR / "ablation_c2.jsonl"
    with open(output_file, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    
    # Summary
    passed = sum(1 for r in results if r.get("test_passed"))
    logger.info("=" * 50)
    logger.info("SUMMARY (Architecture C2 - No S Model)")
    logger.info("Pass rate: %d/%d (%.1f%%)", passed, total, 100 * passed / total if total else 0)
    logger.info("Results saved to: %s", output_file)
    
    return results

In [7]:
# Run the benchmark with Architecture C2
results_c2 = run_humaneval_benchmark(limit=164, shuffle=False)

Loading HumanEval dataset...


Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

2026-01-28 09:42:12,972 | INFO | Loaded 164 tasks from HumanEval (Architecture C2 - No S Model)
2026-01-28 09:42:12,973 | INFO | [1/164] Running task HumanEval/0


Loaded 164 tasks.


2026-01-28 09:42:34,032 | INFO | [1/164] HumanEval/0 PASS in 21.06s (tier=L, escalations=1)
2026-01-28 09:42:34,033 | INFO | [2/164] Running task HumanEval/1
2026-01-28 09:42:54,085 | INFO | [2/164] HumanEval/1 FAIL in 20.05s (tier=L, escalations=1)
2026-01-28 09:42:54,086 | INFO | [3/164] Running task HumanEval/2
2026-01-28 09:43:00,627 | INFO | [3/164] HumanEval/2 PASS in 6.54s (tier=S, escalations=0)
2026-01-28 09:43:00,627 | INFO | [4/164] Running task HumanEval/3
2026-01-28 09:43:15,746 | INFO | [4/164] HumanEval/3 PASS in 15.12s (tier=L, escalations=1)
2026-01-28 09:43:15,747 | INFO | [5/164] Running task HumanEval/4
2026-01-28 09:43:26,922 | INFO | [5/164] HumanEval/4 PASS in 11.17s (tier=M, escalations=1)
2026-01-28 09:43:26,922 | INFO | [6/164] Running task HumanEval/5
2026-01-28 09:43:44,251 | INFO | [6/164] HumanEval/5 FAIL in 17.33s (tier=L, escalations=1)
2026-01-28 09:43:44,252 | INFO | [7/164] Running task HumanEval/6
2026-01-28 09:43:52,293 | INFO | [7/164] HumanEval/6 

---
## Results Analysis

In [8]:
import pandas as pd

# Load C2 results
df_c2 = pd.DataFrame(results_c2)

# Display summary statistics
print("Architecture C2 (No S Model) Results")
print("=" * 40)
print(f"Total tasks: {len(df_c2)}")
print(f"Passed: {df_c2['test_passed'].sum()}")
print(f"Pass rate: {df_c2['test_passed'].mean() * 100:.1f}%")
print(f"Average escalations: {df_c2['escalations'].mean():.2f}")
print(f"Average time: {df_c2['elapsed_seconds'].mean():.2f}s")
print()
print("Developer tier distribution:")
print(df_c2['developer_tier'].value_counts())

Architecture C2 (No S Model) Results
Total tasks: 164
Passed: 113
Pass rate: 68.9%
Average escalations: 0.59
Average time: 13.98s

Developer tier distribution:
developer_tier
L    71
M    71
S    22
Name: count, dtype: int64


In [9]:
# Static Code Quality Metrics (Radon)
# Calculates Cyclomatic Complexity and Maintainability Index for generated code

from radon.complexity import cc_visit
from radon.metrics import mi_visit

def calculate_static_metrics(code: str) -> dict:
    """Calculate static code quality metrics using Radon."""
    if not code or not code.strip():
        return {
            "cyclomatic_complexity_avg": None, 
            "cyclomatic_complexity_max": None,
            "maintainability_index": None
        }
    
    try:
        cc_results = cc_visit(code)
        if cc_results:
            avg_cc = sum(block.complexity for block in cc_results) / len(cc_results)
            max_cc = max(block.complexity for block in cc_results)
        else:
            avg_cc = 1
            max_cc = 1
    except Exception:
        avg_cc = None
        max_cc = None
    
    try:
        mi_score = mi_visit(code, multi=False)
    except Exception:
        mi_score = None
    
    return {
        "cyclomatic_complexity_avg": avg_cc,
        "cyclomatic_complexity_max": max_cc,
        "maintainability_index": mi_score
    }

# Calculate metrics for all generated code
print("\nCalculating static code quality metrics...")
static_metrics = []
for idx, row in df_c2.iterrows():  
    code = row.get("generated_code", "")
    metrics = calculate_static_metrics(code)
    metrics["task_id"] = row["task_id"]
    metrics["test_passed"] = row["test_passed"]
    static_metrics.append(metrics)

metrics_df = pd.DataFrame(static_metrics)

# Add to main dataframe
df_c2["cyclomatic_complexity_avg"] = metrics_df["cyclomatic_complexity_avg"]  
df_c2["cyclomatic_complexity_max"] = metrics_df["cyclomatic_complexity_max"]  
df_c2["maintainability_index"] = metrics_df["maintainability_index"]  

# Summary statistics
valid_cc = metrics_df["cyclomatic_complexity_avg"].dropna()
valid_mi = metrics_df["maintainability_index"].dropna()

print("\n" + "=" * 60)
print("STATIC CODE QUALITY METRICS")
print("=" * 60)
print(f"\nCyclomatic Complexity (lower is better):")
print(f"  Average CC: {valid_cc.mean():.2f}" if len(valid_cc) > 0 else "  Average CC: N/A")
print(f"  Median CC: {valid_cc.median():.2f}" if len(valid_cc) > 0 else "  Median CC: N/A")
print(f"  Max CC: {metrics_df['cyclomatic_complexity_max'].max():.2f}" if metrics_df['cyclomatic_complexity_max'].notna().any() else "  Max CC: N/A")
print(f"\nMaintainability Index (0-100, higher is better):")
print(f"  Average MI: {valid_mi.mean():.2f}" if len(valid_mi) > 0 else "  Average MI: N/A")
print(f"  Median MI: {valid_mi.median():.2f}" if len(valid_mi) > 0 else "  Median MI: N/A")
print(f"  Min MI: {valid_mi.min():.2f}" if len(valid_mi) > 0 else "  Min MI: N/A")
print("=" * 60)

# Passed vs Failed comparison
passed_df = metrics_df[metrics_df["test_passed"] == True]
failed_df = metrics_df[metrics_df["test_passed"] == False]

print(f"\nComparison - Passed vs Failed Tasks:")
print(f"  Passed tasks - Avg CC: {passed_df['cyclomatic_complexity_avg'].mean():.2f}, Avg MI: {passed_df['maintainability_index'].mean():.2f}" if len(passed_df) > 0 else "  Passed tasks: No data")
print(f"  Failed tasks - Avg CC: {failed_df['cyclomatic_complexity_avg'].mean():.2f}, Avg MI: {failed_df['maintainability_index'].mean():.2f}" if len(failed_df) > 0 else "  Failed tasks: No data")



Calculating static code quality metrics...

STATIC CODE QUALITY METRICS

Cyclomatic Complexity (lower is better):
  Average CC: 3.76
  Median CC: 3.00
  Max CC: 14.00

Maintainability Index (0-100, higher is better):
  Average MI: 84.64
  Median MI: 88.47
  Min MI: 50.72

Comparison - Passed vs Failed Tasks:
  Passed tasks - Avg CC: 3.88, Avg MI: 82.75
  Failed tasks - Avg CC: 3.25, Avg MI: 92.26


In [10]:
# Compare with Architecture C baseline if results exist
baseline_file = LOG_DIR / "architecture_C.jsonl"

if baseline_file.exists():
    with open(baseline_file, "r") as f:
        baseline_results = [json.loads(line) for line in f]
    
    df_c = pd.DataFrame(baseline_results)
    
    print("\nComparison: C vs C2")
    print("=" * 50)
    comparison = pd.DataFrame({
        "Metric": ["Pass Rate", "Avg Escalations", "Avg Time (s)"],
        "C (with S)": [
            f"{df_c['test_passed'].mean() * 100:.1f}%",
            f"{df_c['escalations'].mean():.2f}",
            f"{df_c['elapsed_seconds'].mean():.2f}"
        ],
        "C2 (no S)": [
            f"{df_c2['test_passed'].mean() * 100:.1f}%",
            f"{df_c2['escalations'].mean():.2f}",
            f"{df_c2['elapsed_seconds'].mean():.2f}"
        ]
    })
    print(comparison.to_string(index=False))
else:
    print("\nNote: Run architecture-c.ipynb first to get baseline results for comparison.")


Note: Run architecture-c.ipynb first to get baseline results for comparison.


In [11]:
# View raw log output
!cat {LOG_DIR}/ablation_c2.jsonl

{"task_id": "HumanEval/0", "architecture": "C2", "test_passed": true, "escalations": 1, "story_points_initial": 3, "story_points_current": 8, "developer_tier": "L", "elapsed_seconds": 21.06, "generated_code": "from typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    \"\"\" Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    \n    Args:\n        numbers (List[float]): A list of floating-point numbers.\n        threshold (float): The threshold value to compare the closeness of numbers.\n    \n    Returns:\n        bool: True if any two numbers are closer than the threshold, False otherwise.\n    \n    Examples:\n        >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n        False\n        >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n        True\n    \"\"\"\n    numbers.sort()\n    for i in range(len(numbers) - 1):\n        if abs(numbers[i] - numbers[i + 1]) < threshold: